# Chapter 4 · Time Series with Linear Models

Turn a recorded signal into a table, fit a simple model on its own past, and test honestly whether it can forecast.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/nb-04-time-series.ipynb)

- Lecture: [en-04-time-series.md](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-04-time-series.md)
- Lab: [LM401](https://rathachai.github.io/DA-LAB/learnings/linear/lm401.html)

**The 11-step recipe** (same every time): 1 Load libraries · 2 Get data · 3 Display data · 4 Visualize correlation · 5 Process data · 6 Select X and y · 7 Split into train and test · 8 Create and train the model · 9 Predict · 10 Evaluate · 11 Visualize y and y_pred.
Then three short extras: (A) multi-step forecast, (B) window size, (C) random vs chronological split.

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io

## Step 1 · Load libraries
pandas holds tables, numpy does maths, scikit-learn has the model and the scores, matplotlib only draws. Run the cells from top to bottom.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             mean_absolute_percentage_error, r2_score)
plt.rcParams["figure.figsize"] = (9, 4)
print("Libraries ready")

## Step 2 · Get data
`lm401_timeseries.csv` has a time index `t` and ten 200-point signals (like channels from a data logger). We read it straight from the web.

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm401_timeseries.csv"
df = pd.read_csv(url)
print("Columns:", list(df.columns))

## Step 3 · Display data
Look at the first rows, the table size, and a summary (typical level = mean, spread = std, smallest and largest value).

In [ ]:
print("shape (rows, columns):", df.shape)
df.head()

In [ ]:
df.describe().round(2)

## Step 4 · Visualize correlation
First, look at the signals: trend, repeating season, a random walk that wanders, and white noise with no pattern. Then ask: *is today's value related to yesterday's?* If yes, the past can predict the future.

In [ ]:
names = [col for col in df.columns if col != "t"]
fig, axes = plt.subplots(5, 2, figsize=(11, 10), sharex=True)
for ax, name in zip(axes.ravel(), names):
    ax.plot(df["t"], df[name], lw=1.2)
    ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
s = df["trend_season"]
r = s.corr(s.shift(1))                      # correlation of y(t) with y(t-1)
plt.scatter(s.shift(1), s, s=12)
plt.title(f"trend_season: y(t) vs y(t-1), r = {r:.3f}")
plt.xlabel("y(t-1)  (yesterday)"); plt.ylabel("y(t)  (today)"); plt.show()

Autocorrelation = the same correlation for several lags (how many steps back). High bars mean the past is useful. Compare a trending season with a random walk and pure noise.

In [ ]:
lags = range(1, 7)
ac = pd.DataFrame({n: [df[n].corr(df[n].shift(k)) for k in lags]
                   for n in ["trend_season", "random_walk", "white_noise"]}, index=lags)
ac.index.name = "lag"
display(ac.round(3))
ac.plot.bar(rot=0); plt.title("Autocorrelation by lag"); plt.ylabel("r"); plt.show()

## Step 5 · Process data
A series is **one array** `y0, y1, ..., yn` (`n+1` values) in time order. There is no input column and no answer column, so we build them with a **sliding window** of size `w`: the inputs are the previous `w` values (`x1` oldest ... `xw` newest) and the answer `y` is the value that comes next. In pandas, `shift(k)` slides a column down by `k` rows, which gives us the past. The first `w` rows have no full window and are dropped, so we get **`n+1-w` rows**. Here `w = 5` on `trend_season`.

In [ ]:
w = 5
s = df["trend_season"]
df_w = pd.DataFrame({f"x{k}": s.shift(w - k + 1) for k in range(1, w + 1)})  # x1 = shift(5) ... x5 = shift(1)
df_w["y"] = s                       # the value that comes next
df_w = df_w.dropna()
df_w.index.name = "t"               # t = time index of the target
df_w.head()

In [ ]:
print("df rows n+1 =", len(df), "| window w =", w, "| df_w rows n+1-w =", len(df_w))
print("df_w shape:", df_w.shape, "(w inputs + 1 target)")

Check against the ready-made table the course ships (`lm401_window_w5_trend_season.csv`).

In [ ]:
url_w = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm401_window_w5_trend_season.csv"
ready = pd.read_csv(url_w).set_index("t")
cols = [f"x{k}" for k in range(1, w + 1)] + ["y"]
print("same shape:", ready[cols].shape == df_w.shape)
print("same values:", np.allclose(df_w[cols].to_numpy(), ready[cols].to_numpy()))

## Step 6 · Select X and y
`X` = the inputs (the last 5 values), `y` = the answer (the next value).

In [ ]:
feats = [f"x{k}" for k in range(1, w + 1)]
X = df_w[feats]
y = df_w["y"]
print("X:", X.shape, " y:", y.shape)
X.head(3)

## Step 7 · Split into train and test
We train on the **early** 70 % and test on the **late** 30 %. We do not shuffle: real forecasting predicts the future from the past, and neighbouring rows share most of their numbers, so a random split would let the model peek at near-copies of the test rows (leakage; see Section C).

In [ ]:
n_train = int(round(0.7 * len(df_w)))
X_train, X_test = X.iloc[:n_train], X.iloc[n_train:]
y_train, y_test = y.iloc[:n_train], y.iloc[n_train:]
print("train rows:", len(X_train), " test rows:", len(X_test))
print("same as the ready-made split column:", (ready["split"].iloc[:n_train] == "train").all()
      and (ready["split"].iloc[n_train:] == "test").all())

In [ ]:
plt.plot(y_train.index, y_train, label="train (early)")
plt.plot(y_test.index, y_test, lw=2.5, label="test (late)")
plt.axvline(y_test.index[0], color="gray", ls="--")
plt.title("Chronological split of trend_season"); plt.xlabel("time index t")
plt.ylabel("value"); plt.legend(); plt.show()

## Step 8 · Create and train the model
`LinearRegression` learns one weight per input: `y_hat = b0 + b1*x1 + ... + b5*x5`. This is an **autoregressive (AR)** model: a signal predicted from its own past, like a difference equation whose coefficients come from data. We fit on the training rows only.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
eq = " + ".join(f"{b:.3f}*x{k}" for k, b in enumerate(model.coef_, 1))
print(f"y_hat = {model.intercept_:.3f} + {eq}")
print("x5 is the newest sample, x1 the oldest.")

## Step 9 · Predict
**One step ahead**: at every time the model is given the *real* last 5 values and predicts only the next one, so errors do not pile up.

In [ ]:
y_pred = model.predict(X_test)
pd.DataFrame({"actual": y_test, "y_pred": y_pred}).round(2).head()

## Step 10 · Evaluate
MAE = average size of the error. RMSE = like MAE but punishes big misses. MAPE = error in percent. R² = share of the ups and downs explained (1 is perfect). The **persistence baseline** ("next value = last value", i.e. `x5`) is the trivial rule any useful model must beat.

In [ ]:
def metrics(yt, yp):
    return {"MAE": mean_absolute_error(yt, yp),
            "RMSE": mean_squared_error(yt, yp) ** 0.5,
            "MAPE %": 100 * mean_absolute_percentage_error(yt, yp),
            "R2": r2_score(yt, yp)}
res = pd.DataFrame({"train": metrics(y_train, model.predict(X_train)),
                    "test": metrics(y_test, y_pred),
                    "test, persistence baseline": metrics(y_test, X_test["x5"])}).T
res.round(3)

## Step 11 · Visualize y and y_pred
Left: actual and predicted over time (test region shaded). Right: actual vs predicted; points on the diagonal are perfect.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(y.index, y, color="gray", label="actual")
a1.plot(y_test.index, y_pred, color="tab:red", label="one-step y_pred")
a1.axvspan(y_test.index[0], y_test.index[-1], color="tab:blue", alpha=0.12, label="test")
a1.set_title("trend_season over time"); a1.set_xlabel("t"); a1.legend()
a2.scatter(y_test, y_pred, s=14)
lim = [y_test.min(), y_test.max()]
a2.plot(lim, lim, "k--"); a2.set_title("Actual vs predicted (test)")
a2.set_xlabel("actual"); a2.set_ylabel("predicted")
plt.tight_layout(); plt.show()

## A · Multi-step (recursive) forecast
Now pretend there are **no new measurements** in the test period. Start from the last 5 known values, predict one step, feed the prediction back in as input, and repeat (dead reckoning). Errors can now build up.

In [ ]:
window = list(X_test.iloc[0])               # last 5 real values before the test period
rec = []
for _ in range(len(X_test)):
    nxt = model.predict(pd.DataFrame([window], columns=feats))[0]
    rec.append(nxt)
    window = window[1:] + [nxt]               # drop oldest, add the prediction
rec = np.array(rec)
pd.DataFrame({"one-step": metrics(y_test, y_pred), "recursive": metrics(y_test, rec)}).T.round(3)

In [ ]:
plt.plot(y_train.index, y_train, color="lightgray", label="train")
plt.plot(y_test.index, y_test, color="black", lw=2, label="actual")
plt.plot(y_test.index, y_pred, color="tab:red", label="one-step")
plt.plot(y_test.index, rec, color="tab:blue", ls="--", lw=2, label="recursive")
plt.title("Actual vs one-step vs recursive"); plt.xlabel("t"); plt.legend(); plt.show()

The one-step line hugs the actual signal; the recursive line drifts away because it never sees real data again.

## B · Window size
Is a bigger window always better? Try `w = 1..10` on `trend_season` and `sine` and compare the one-step test MAE (chronological split).

In [ ]:
def make_window(series, w):
    d = pd.DataFrame({f"x{k}": series.shift(w - k + 1) for k in range(1, w + 1)})
    d["y"] = series
    return d.dropna()

def fit_eval(name, w):
    d = make_window(df[name], w); k = int(round(0.7 * len(d)))
    X_, y_ = d.drop(columns="y"), d["y"]
    m = LinearRegression().fit(X_.iloc[:k], y_.iloc[:k])
    return mean_absolute_error(y_.iloc[k:], m.predict(X_.iloc[k:]))

In [ ]:
piv = pd.DataFrame({n: [fit_eval(n, ww) for ww in range(1, 11)]
                    for n in ["trend_season", "sine"]}, index=pd.Index(range(1, 11), name="w"))
display(piv.round(3))
piv.plot(marker="o"); plt.title("Test MAE vs window size w")
plt.ylabel("test MAE"); plt.show()

A sine wave is fully described by 2 lags (position and direction), so most of the gain comes early; the rest is noise. For `trend_season` the error does not fall steadily, so more lags are not automatically better.

## C · Random vs chronological split
If we shuffle the rows, each test row is almost a copy of a training row, so the error can look better than it honestly is. We compare both splits (random split averaged over 20 shuffles) with `w = 5`.

In [ ]:
def split_errors(name, w=5):
    d = make_window(df[name], w); X_, y_ = d.drop(columns="y"), d["y"]
    k = int(round(0.7 * len(d)))
    m = LinearRegression().fit(X_.iloc[:k], y_.iloc[:k])
    chrono = mean_absolute_error(y_.iloc[k:], m.predict(X_.iloc[k:]))
    rnd = []
    for seed in range(20):
        Xa, Xb, ya, yb = train_test_split(X_, y_, test_size=0.3, random_state=seed)
        rnd.append(mean_absolute_error(yb, LinearRegression().fit(Xa, ya).predict(Xb)))
    return {"chronological MAE": chrono, "random MAE": np.mean(rnd)}

In [ ]:
leak = pd.DataFrame({n: split_errors(n) for n in ["trend_season", "ar1", "random_walk", "sine"]}).T
display(leak.round(3))
leak.plot.bar(rot=0); plt.title("Test MAE: chronological vs random split"); plt.ylabel("test MAE"); plt.show()

The random split reports a different (often smaller) error than the honest chronological one. A simple linear model leaks only a little; flexible models suffer far more. Rule: for forecasting, always split by time.

## Summary
- A time series is **one array** in time order; a **sliding window** (`shift`) turns it into a table with `n+1-w` rows, and a linear model on lags is an **AR(w)** model.
- **Split chronologically** (early = train, late = test); random splits leak the future.
- **One-step** predictions use real data; **recursive** forecasts reuse their own predictions, so error grows with the horizon.
- Always compare with the **persistence baseline**; for a random walk it is hard to beat. Bigger `w` is not always better.

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io